# 基于GRPO的Qwen3.5小模型理科推理能力增强研究
## Kaggle Notebook — 在 2×T4 上运行完整实验

## Step 1: 安装依赖

In [ ]:
!pip install -q transformers>=4.47.0 datasets>=3.0.0 accelerate>=1.0.0
!pip install -q peft>=0.13.0 bitsandbytes>=0.44.0 trl>=0.12.0
!pip install -q wandb tensorboard gradio>=5.0.0
!pip install -q numpy pandas matplotlib seaborn tqdm

## Step 2: 导入 + 检查 GPU

In [ ]:
import os
import sys
import torch
import json

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB')

## Step 3: 设置路径 + 克隆参考代码

In [ ]:
# 创建项目目录
!mkdir -p /kaggle/working/project/src
!mkdir -p /kaggle/working/project/outputs
!mkdir -p /kaggle/working/project/data

# 复制 reward.py
%%writefile /kaggle/working/project/src/reward.py
# [将 reward.py 内容粘贴到此处，或从 Git 克隆]

## Step 4: 加载模型 + Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen3.5-0.8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)
print('Model loaded!')

## Step 5: Baseline ① — 基座模型 (no thinking)

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import re

def extract_answer(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    m = re.search(r'(?:答案|所以|因此)[：:]\s*(.+)', text.strip())
    return m.group(1).strip() if m else text.strip()[:100]

def evaluate_baseline(model, tokenizer, enable_thinking=True, max_samples=200):
    dataset = load_dataset("openai/gsm8k", "main", split="test")
    correct = 0
    results = []
    
    for i in tqdm(range(min(max_samples, len(dataset)))):
        ex = dataset[i]
        question = ex['question']
        gt = ex['answer'].split('####')[-1].strip()
        
        sys_prompt = "一步步思考" if enable_thinking else "直接给答案"
        prompt = f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
        response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        pred = extract_answer(response)
        is_correct = (pred.strip() == gt.strip())
        if is_correct:
            correct += 1
        results.append({'question': question, 'gt': gt, 'pred': pred, 'correct': is_correct})
    
    acc = correct / len(results) * 100
    print(f'Accuracy: {acc:.1f}% ({correct}/{len(results)})')
    return acc, results

print('Baseline ①: No thinking')
acc1, res1 = evaluate_baseline(model, tokenizer, enable_thinking=False)

## Step 6: Baseline ② — 基座模型 (thinking on)

In [ ]:
print('Baseline ②: With thinking')
acc2, res2 = evaluate_baseline(model, tokenizer, enable_thinking=True)

## Step 7: SFT 预热训练

In [ ]:
# 调用 train_sft.py
!python /kaggle/working/project/src/train_sft.py \
    --model_name Qwen/Qwen3.5-0.8B-Instruct \
    --output_dir /kaggle/working/sft_output \
    --num_epochs 2 \
    --per_device_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --lora_r 16 \
    --max_seq_length 1024 \
    --max_train_samples 4000

## Step 8: GRPO 主实验

In [ ]:
!python /kaggle/working/project/src/train_grpo.py \
    --model_name Qwen/Qwen3.5-0.8B-Instruct \
    --sft_adapter_path /kaggle/working/sft_output/final \
    --output_dir /kaggle/working/grpo_main \
    --reward_type full \
    --num_generations 8 \
    --kl_coef 0.04 \
    --learning_rate 5e-5 \
    --num_train_epochs 2 \
    --per_device_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_train_samples 2000

## Step 9: 消融实验

In [ ]:
# 消融 A: 无SFT预热 — 直接GRPO
!python /kaggle/working/project/src/train_grpo.py \
    --model_name Qwen/Qwen3.5-0.8B-Instruct \
    --output_dir /kaggle/working/grpo_abl_no_sft \
    --reward_type full \
    --num_generations 8 \
    --kl_coef 0.04 \
    --num_train_epochs 2 \
    --per_device_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_train_samples 2000

In [ ]:
# 消融 B: 仅正确性奖励
!python /kaggle/working/project/src/train_grpo.py \
    --model_name Qwen/Qwen3.5-0.8B-Instruct \
    --sft_adapter_path /kaggle/working/sft_output/final \
    --output_dir /kaggle/working/grpo_abl_correctness \
    --reward_type correctness_only \
    --num_generations 8 \
    --kl_coef 0.04 \
    --num_train_epochs 2 \
    --per_device_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_train_samples 2000

## Step 10: 评估所有模型

In [ ]:
import json
results = {
    'baseline_no_thinking': acc1,
    'baseline_thinking': acc2,
    # 添加 SFT 和 GRPO 评估结果
}

# 保存
with open('/kaggle/working/all_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# 对比柱状图
import matplotlib.pyplot as plt
import numpy as np

labels = list(results.keys())
values = list(results.values())

plt.figure(figsize=(12, 6))
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6']
plt.bar(labels, values, color=colors[:len(labels)])
plt.xlabel('Method')
plt.ylabel('Accuracy (%)')
plt.title('Qwen3.5-0.8B Reasoning Performance Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/kaggle/working/results_comparison.png', dpi=150)
plt.show()